# 09 - Day 5: Cross-Dataset Validation (CSE-CIC-IDS2018)

**Naming note:** the brief requested `notebooks/08_day5_cross_dataset_validation.ipynb`,
but `08_day4_explainability_unseen_attack_analysis.ipynb` already occupies
`08` in this project's numbering. This notebook is filed as `09_` to
avoid overwriting Day 4's notebook, following the project's existing
increment-per-day convention. Let me know if a different name is preferred.

## Research question

**Does the NIDS generalize when evaluated on traffic from a different dataset/distribution?**

This notebook evaluates the already-frozen Day 1 Random Forest, Day 2
IsolationForest, and Day 2 hybrid detector on an external dataset,
using CIC-IDS2017's frozen thresholds. Nothing is retrained. No
threshold is tuned on external data.

## Dataset decision

No project-proposal document was available to inspect in this
environment. In its absence, **CSE-CIC-IDS2018** was assumed as the
external dataset, since it is the standard, widely-used follow-up
dataset to CIC-IDS2017 for exactly this kind of cross-dataset NIDS
generalization study (same research group -- Canadian Institute for
Cybersecurity -- similar CICFlowMeter-derived feature set, different
capture environment/time period). If your proposal specifies a
different dataset, re-run with `--external-dir` pointed at it; the
feature-mapping layer below is dataset-agnostic as long as the target
dataset also exposes CICFlowMeter-style flow features and a label
column.

## Critical scientific rules enforced here

- The external dataset is used **only** for evaluation and
  distribution-shift comparison -- never for fitting, threshold
  selection, or hyperparameter tuning.
- `select_threshold_from_validation` is never imported in this
  notebook or in `scripts/run_day5.py` (verified by a dedicated test).
- `check_is_fitted` runs on both loaded models before any external
  data is touched, proving they are pre-fitted artifacts, not refit here.
- If the external dataset is not found, this notebook does **not**
  fabricate metrics -- it reports `status: "pipeline_ready_dataset_required"`
  and stops.


## 1. Day 5 objective

Evaluate cross-dataset generalization of the frozen Day 1/2 detectors, with an honest, non-forced conclusion -- including if the Hybrid detector performs worse, which Day 4 already showed happens under temporal shift within CIC-IDS2017 itself.

## 2. Research question

Does the NIDS generalize when evaluated on traffic from a different dataset/distribution? Compared: Random Forest, IsolationForest, Hybrid.

## 3. External dataset

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve
from sklearn.utils.validation import check_is_fitted

from src.data.validator import detect_label_column
from src.day2.anomaly import AnomalyModel, anomaly_scores
from src.day2.hybrid import HybridConfig, combine_scores, hybrid_predict
from src.day2.thresholding import evaluate_frozen_threshold, per_class_frozen_results
from src.day4.analysis import distribution_shift_tests, rf_feature_importances, top_n_features
from src.day5.feature_mapping import apply_feature_mapping, build_feature_mapping
from src.day5.label_mapping import map_external_labels

DATA_DIR = PROJECT_ROOT / "data" / "processed" / "day1"
DAY1_MODEL_DIR = PROJECT_ROOT / "models" / "day1"
DAY2_MODEL_DIR = PROJECT_ROOT / "models" / "day2"
DAY2_RESULTS_DIR = PROJECT_ROOT / "results" / "day2"
RESULTS_DIR = PROJECT_ROOT / "results" / "day5"
FIGURES_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "CSE-CIC-IDS2018"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external" / "cse_cic_ids2018"
RF_FROZEN_THRESHOLD = 0.01
FALLBACK_IF_THRESHOLD = 0.15
FALLBACK_HYBRID_THRESHOLD = 0.50
TOP_N_FEATURES_FOR_SHIFT = 20
SAMPLE_CAP = 20_000
RANDOM_STATE = 42

print(f"Expected external dataset location: {EXTERNAL_DIR}")
print("No model is retrained and no threshold is tuned using external data.")


## 4. Dataset availability

Checks for `*.csv` files in the expected location. If none are found, this notebook writes an honest `pipeline_ready_dataset_required` status and stops -- it does not fabricate metrics.

In [ ]:
required = {
    "train": DATA_DIR / "train.parquet",
    "metadata": DATA_DIR / "split_metadata.json",
    "rf_model": DAY1_MODEL_DIR / "random_forest_baseline.joblib",
    "if_model": DAY2_MODEL_DIR / "isolation_forest.joblib",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"Day 5 requires existing Day 1/Day 2 artifacts. Missing: {missing}")

metadata = json.loads(required["metadata"].read_text())
feature_names = metadata["feature_names"]

train_df = pd.read_parquet(required["train"])
rf_model = joblib.load(required["rf_model"])
if_raw_model = joblib.load(required["if_model"])

check_is_fitted(rf_model)
check_is_fitted(if_raw_model)
print("Integrity check passed: RF and IsolationForest are pre-fitted, loaded models.")

train_medians = train_df[feature_names].median(numeric_only=True)

external_csvs = sorted(EXTERNAL_DIR.glob("*.csv")) if EXTERNAL_DIR.exists() else []
print(f"External CSV files found: {len(external_csvs)}")


In [ ]:
day2_threshold_path = DAY2_RESULTS_DIR / "validation_threshold_selection.json"
rf_threshold = RF_FROZEN_THRESHOLD
if day2_threshold_path.exists():
    day2_thresholds = json.loads(day2_threshold_path.read_text())
    if_threshold = day2_thresholds["isolation_forest"]["selected_threshold"]
    hybrid_threshold = day2_thresholds["hybrid"]["selected_threshold"]
    threshold_source = str(day2_threshold_path)
else:
    print(f"WARNING: {day2_threshold_path} not found -- using documented fallback thresholds.")
    if_threshold = FALLBACK_IF_THRESHOLD
    hybrid_threshold = FALLBACK_HYBRID_THRESHOLD
    threshold_source = "fallback_default"
hybrid_config = HybridConfig(threshold=hybrid_threshold)

print(f"RF={rf_threshold}, IF={if_threshold}, Hybrid={hybrid_threshold} (source: {threshold_source})")

if not external_csvs:
    status = {
        "status": "pipeline_ready_dataset_required",
        "dataset_name": DATASET_NAME,
        "expected_location": str(EXTERNAL_DIR),
        "exact_command_to_rerun": f"python scripts/run_day5.py --external-dir {EXTERNAL_DIR}",
        "note": "feature_mapping.json and label_mapping.json cannot be produced without the actual dataset's columns/labels -- not fabricated here.",
    }
    (RESULTS_DIR / "cross_dataset_metrics.json").write_text(json.dumps(status, indent=2, default=str))
    print(json.dumps(status, indent=2))
    print("\nDay 5 pipeline is implemented and ready, but the experiment was NOT experimentally completed --", DATASET_NAME, "was not found.")
    print("Remaining cells below require the dataset to be present; stop here if it is not.")


## 5. Feature compatibility

Matches CIC-IDS2017 feature names against the external dataset's raw columns by a version-independent normalized token key (handles renames like `Destination Port` vs `Dst Port`, `Bwd Packets/s` vs `Bwd Pkts/s`). Every unmapped CIC-IDS2017 feature is recorded explicitly and imputed with a CIC-IDS2017 TRAINING statistic -- never silently dropped, never derived from external data.

In [ ]:
assert external_csvs, "No external CSVs found -- see Section 4 above."

external_df = pd.concat([pd.read_csv(p, low_memory=False) for p in external_csvs], ignore_index=True)
external_df.columns = [c.strip() for c in external_df.columns]
print(f"External dataset loaded: {len(external_df)} rows, {external_df.shape[1]} columns.")


## 6. Label mapping

Reuses `src.data.cleaner.normalize_labels` (unchanged) to derive `label_binary` (Benign=0/Attack=1) and preserve the external dataset's own attack-family names in `label_multiclass`.

In [ ]:
label_col = detect_label_column(list(external_df.columns))
assert label_col is not None, "Could not detect a label column in the external dataset."

external_df, label_mapping_table, _log = map_external_labels(external_df, label_col=label_col)
label_mapping_table.to_csv(RESULTS_DIR / "label_mapping_table.csv", index=False)
(RESULTS_DIR / "label_mapping.json").write_text(json.dumps({
    "external_label_column_detected": label_col,
    "benign_aliases": ["benign"],
    "mapping_table": label_mapping_table.to_dict(orient="records"),
}, indent=2, default=str))
label_mapping_table


## 7. Preprocessing (feature mapping applied)

In [ ]:
mapping = build_feature_mapping(feature_names, external_df.columns)
(RESULTS_DIR / "feature_mapping.json").write_text(json.dumps(mapping.summary(), indent=2, default=str))

print(f"{len(mapping.mapped)}/{len(feature_names)} CIC-IDS2017 features mapped.")
if mapping.unmapped_cic2017_features:
    print("Unmapped (imputed with training medians):", mapping.unmapped_cic2017_features)

external_mapped = apply_feature_mapping(external_df, mapping, train_medians).fillna(train_medians)
assert not external_mapped.isna().any().any()
assert not np.isinf(external_mapped.to_numpy()).any()
external_mapped.head()


## 8. Frozen model configuration

RF/IsolationForest/Hybrid thresholds and hybrid weighting are reused exactly as frozen in Day 1/Day 2 -- nothing is re-selected here.

In [ ]:
if_config = {}
day2_metadata_path = DAY2_MODEL_DIR / "day2_metadata.json"
if day2_metadata_path.exists():
    if_config = json.loads(day2_metadata_path.read_text()).get("isolation_forest_config", {})

anomaly_model = AnomalyModel(
    model=if_raw_model, feature_names=list(feature_names), train_medians=train_medians,
    contamination=if_config.get("contamination", if_raw_model.get_params().get("contamination")),
    n_estimators=if_config.get("n_estimators", if_raw_model.get_params().get("n_estimators")),
    random_state=if_config.get("random_state", if_raw_model.get_params().get("random_state")),
    n_training_samples=if_config.get("n_training_samples", -1),
)

ext_proba = rf_model.predict_proba(external_mapped[feature_names])[:, 1]
ext_anomaly = anomaly_scores(anomaly_model, external_mapped)
ext_hybrid = combine_scores(ext_proba, ext_anomaly, hybrid_config)

y_ext = external_df["label_binary"].astype(int).to_numpy()
detector_scores = {
    "random_forest": (ext_proba, rf_threshold),
    "isolation_forest": (ext_anomaly, if_threshold),
    "hybrid": (ext_hybrid, hybrid_threshold),
}
print("Scored", len(external_df), "external rows with RF, IsolationForest, and Hybrid.")


## 9-11. Random Forest / IsolationForest / Hybrid evaluation

Reuses `src.day2.thresholding.evaluate_frozen_threshold` (unmodified) for precision, recall, F1, FPR, FNR, confusion matrix, ROC-AUC, PR-AUC.

In [ ]:
n_total = len(external_df)
n_benign = int((y_ext == 0).sum())
n_attack = int((y_ext == 1).sum())

comparison_rows = []
for detector_name, (scores, threshold) in detector_scores.items():
    metrics = evaluate_frozen_threshold(y_ext, scores, threshold)
    comparison_rows.append({
        "detector": detector_name, "dataset": DATASET_NAME, "threshold_used": threshold,
        "total_samples": n_total, "benign_samples": n_benign, "attack_samples": n_attack,
        "attack_detection_rate": metrics["recall"], **metrics,
    })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(RESULTS_DIR / "comparison_table.csv", index=False)
comparison


In [ ]:
family_labels = external_df["label_multiclass"]
attack_family_results = None
if family_labels[family_labels != "Benign"].nunique() >= 1:
    family_tables = []
    for detector_name, (scores, threshold) in detector_scores.items():
        preds = (np.asarray(scores) >= threshold).astype(int)
        table = per_class_frozen_results(scores, preds, family_labels)
        table.insert(0, "detector", detector_name)
        family_tables.append(table)
    attack_family_results = pd.concat(family_tables, ignore_index=True)
    attack_family_results.to_csv(RESULTS_DIR / "attack_family_results.csv", index=False)
attack_family_results


## 12. Distribution shift

Reuses `src.day4.analysis.distribution_shift_tests` (unmodified) to compare CIC-IDS2017 training vs. the external dataset on the top RF-important features that were actually mapped (imputed placeholder features are excluded, since comparing a constant to itself would be a degenerate, meaningless '0% shift').

In [ ]:
importances = rf_feature_importances(rf_model, feature_names)
top_features = top_n_features(importances, TOP_N_FEATURES_FOR_SHIFT)
shiftable_features = [f for f in top_features if f in mapping.mapped]
excluded_from_shift = [f for f in top_features if f not in mapping.mapped]
if excluded_from_shift:
    print("Excluded from shift testing (not mappable):", excluded_from_shift)

dist_tests = pd.DataFrame()
if shiftable_features:
    dist_tests = distribution_shift_tests(
        train_df, {"external": external_mapped}, shiftable_features,
        sample_cap=SAMPLE_CAP, random_state=RANDOM_STATE,
    )
dist_tests.to_csv(RESULTS_DIR / "distribution_shift.csv", index=False)
dist_tests.sort_values("p_value").head(15) if not dist_tests.empty else dist_tests


## 13. Score distribution analysis

Whether the frozen thresholds remain meaningful on the external distribution.

In [ ]:
def score_hist(scores, threshold, title, xlabel, out_name):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(np.asarray(scores)[y_ext == 0], bins=40, alpha=0.6, label="Benign", density=True)
    ax.hist(np.asarray(scores)[y_ext == 1], bins=40, alpha=0.6, label="Attack", density=True)
    ax.axvline(threshold, color="black", linestyle="--", label=f"Frozen threshold = {threshold}")
    ax.set_xlabel(xlabel); ax.set_ylabel("Density"); ax.set_title(title); ax.legend()
    plt.tight_layout(); fig.savefig(FIGURES_DIR / out_name, dpi=150); plt.show()

score_hist(ext_proba, rf_threshold, f"RF score distribution -- {DATASET_NAME}", "RF attack probability", "rf_score_distribution.png")


In [ ]:
score_hist(ext_anomaly, if_threshold, f"IsolationForest score distribution -- {DATASET_NAME}", "Normalized anomaly score", "if_score_distribution.png")


In [ ]:
score_hist(ext_hybrid, hybrid_threshold, f"Hybrid score distribution -- {DATASET_NAME}", "Hybrid score", "hybrid_score_distribution.png")


## 14. Detector comparison

Bar chart, ROC curves, PR curves, and confusion matrices for RF vs. IsolationForest vs. Hybrid on the external dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
comparison.set_index("detector")[["precision", "recall", "f1"]].plot(kind="bar", ax=ax)
ax.set_ylim(0, 1.05); ax.set_ylabel("Score"); ax.set_title(f"Detector comparison on {DATASET_NAME} (frozen thresholds)")
plt.xticks(rotation=0); plt.tight_layout()
fig.savefig(FIGURES_DIR / "detector_comparison.png", dpi=150); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for detector_name, (scores, _t) in detector_scores.items():
    fpr, tpr, _ = roc_curve(y_ext, scores)
    ax.plot(fpr, tpr, label=detector_name)
ax.plot([0, 1], [0, 1], linestyle=":", color="gray", label="Chance")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate"); ax.set_title(f"ROC curves -- {DATASET_NAME}")
ax.legend(); plt.tight_layout()
fig.savefig(FIGURES_DIR / "roc_curves.png", dpi=150); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for detector_name, (scores, _t) in detector_scores.items():
    prec, rec, _ = precision_recall_curve(y_ext, scores)
    ax.plot(rec, prec, label=detector_name)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.set_title(f"Precision-Recall curves -- {DATASET_NAME}")
ax.legend(); plt.tight_layout()
fig.savefig(FIGURES_DIR / "pr_curves.png", dpi=150); plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (detector_name, row) in zip(axes, comparison.set_index("detector").iterrows()):
    cm = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]])
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred Benign", "Pred Attack"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["True Benign", "True Attack"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    ax.set_title(detector_name)
fig.suptitle(f"Confusion matrices -- {DATASET_NAME} (frozen thresholds)")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_matrices.png", dpi=150); plt.show()


## 15. Limitations

- Cross-dataset evaluation compares two different capture environments, time periods, and (often) traffic-generation methodologies -- differences in detector performance may reflect any of these, not solely "generalization" in the abstract.
- Any CIC-IDS2017 feature that could not be matched to a real external column was imputed with a training-derived constant, not genuinely observed on the external traffic -- this is a limitation of the common-feature approach, documented explicitly in `feature_mapping.json`, not a property of the external traffic itself.
- No model was retrained. No threshold was tuned using external data. All thresholds and model artifacts are reused exactly as frozen in Day 1/Day 2.
- Statistical significance in distribution-shift tests does not establish that the shift *causes* any specific detector's performance change on this dataset.

## 16. Research interpretation

In [ ]:
rf_row = comparison[comparison["detector"] == "random_forest"].iloc[0]
if_row = comparison[comparison["detector"] == "isolation_forest"].iloc[0]
hybrid_row = comparison[comparison["detector"] == "hybrid"].iloc[0]

findings = [
    f"Random Forest F1 on {DATASET_NAME}: {rf_row['f1']:.4f} (recall={rf_row['recall']:.4f}, precision={rf_row['precision']:.4f}), using the frozen threshold {rf_threshold} without any external-data tuning.",
    f"IsolationForest F1 on {DATASET_NAME}: {if_row['f1']:.4f} (recall={if_row['recall']:.4f}).",
    f"Hybrid F1 on {DATASET_NAME}: {hybrid_row['f1']:.4f} (recall={hybrid_row['recall']:.4f}).",
]
if hybrid_row["f1"] < min(rf_row["f1"], if_row["f1"]):
    findings.append("The hybrid detector performs worse than both individual detectors on this external dataset, consistent with the Day 4 finding that its RF-heavy weighting inherits reduced RF confidence under distribution shift -- reported honestly rather than tuned away.")
elif hybrid_row["f1"] > max(rf_row["f1"], if_row["f1"]):
    findings.append("The hybrid detector improves over both individual detectors on this external dataset.")
else:
    findings.append("The hybrid detector's performance falls between the two individual detectors on this external dataset.")

if not dist_tests.empty:
    n_sig = int((dist_tests["p_value"] < 0.01).sum())
    findings.append(f"{n_sig}/{len(dist_tests)} tested top-importance features show a statistically significant distribution difference (p<0.01) between CIC-IDS2017 training and {DATASET_NAME}. This is reported as an association between distribution differences and detector performance, not as evidence of causation.")
if mapping.unmapped_cic2017_features:
    findings.append(f"{len(mapping.unmapped_cic2017_features)} of {len(feature_names)} CIC-IDS2017 features could not be matched to a real external column and were imputed with training medians for evaluation.")

interpretation = {
    "research_question": "Does the NIDS generalize when evaluated on traffic from a different dataset/distribution?",
    "dataset": DATASET_NAME,
    "findings": findings,
    "caveats": [
        "Cross-dataset evaluation conflates dataset, time-period, and methodology differences -- it does not isolate 'generalization' in the abstract.",
        f"{len(mapping.unmapped_cic2017_features)}/{len(feature_names)} features required imputation rather than being genuinely observed on the external dataset.",
        "No model was retrained. No threshold was tuned using external data.",
    ],
}
(RESULTS_DIR / "cross_dataset_metrics.json").write_text(json.dumps({
    "status": "completed", "dataset_name": DATASET_NAME,
    "comparison_table": comparison.to_dict(orient="records"), "interpretation": interpretation,
}, indent=2, default=str))

print("=== Day 5 Research Interpretation ===")
for line in findings:
    print("-", line)


## 17. Conclusion

This notebook implements and (when the external dataset is present)
executes a complete cross-dataset generalization experiment: a
version-independent feature-name mapping layer, label mapping reusing
Day 1's existing normalization logic, evaluation with Day 1/Day 2's
frozen thresholds and hybrid weighting (never re-tuned), distribution-shift
testing reusing Day 4's implementation, and an honest, non-forced
research interpretation.

If the external dataset was not available when this notebook last ran,
the experiment is **implemented but not experimentally completed** --
`results/day5/cross_dataset_metrics.json` will show
`status: "pipeline_ready_dataset_required"` rather than any metric.
Re-run this notebook (or `python scripts/run_day5.py`) once the dataset
is placed at the documented location for real results.


## Metadata

In [ ]:
day5_metadata = {
    "status": "completed" if external_csvs else "pipeline_ready_dataset_required",
    "dataset_name": DATASET_NAME,
    "external_files_used": [str(p) for p in external_csvs],
    "frozen_thresholds": {"random_forest": rf_threshold, "isolation_forest": if_threshold, "hybrid": hybrid_threshold, "source": threshold_source},
    "hybrid_config": hybrid_config.summary(),
    "feature_mapping_summary": mapping.summary() if external_csvs else None,
    "no_models_retrained": True,
    "no_external_data_used_for_threshold_selection": True,
}
(RESULTS_DIR / "day5_metadata.json").write_text(json.dumps(day5_metadata, indent=2, default=str))
print("Day 5 outputs written to:", RESULTS_DIR)
